# Marginal constraints during Sinkhorn scaling

This notebook generates `fig:sinkhorn-marginal-errors`.  Sinkhorn scaling alternates row and column normalizations of a positive kernel.  The figure shows the current coupling matrix together with the current row and column marginals, so that the projection mechanism is visible before plotting any convergence curve.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.patches import Polygon

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    density = np.zeros_like(x, dtype=float)
    for weight, mean, std in zip(weights, means, stds):
        density += weight * normal_pdf(x, mean, std)
    return density


def sinkhorn_histograms(n=170):
    grid = np.linspace(-3.25, 3.15, n)
    alpha_density = mixture_pdf(grid, [0.58, 0.42], [-1.95, -0.10], [0.34, 0.54])
    beta_density = mixture_pdf(grid, [0.42, 0.58], [-0.75, 1.55], [0.42, 0.36])
    a = alpha_density / alpha_density.sum()
    b = beta_density / beta_density.sum()
    C = (grid[:, None] - grid[None, :]) ** 2
    C = C / np.median(C[C > 0])
    return grid, alpha_density, beta_density, a, b, C


def kl_normalized_solution(a, b, C, epsilon, *, num_iter=10000):
    K = (a[:, None] * b[None, :]) * np.exp(-C / epsilon)
    plan, log = ot.sinkhorn(
        a,
        b,
        K,
        reg=1.0,
        method="sinkhorn_log",
        log=True,
        numItermax=num_iter,
        stopThr=1e-12,
    )
    f = epsilon * np.asarray(log["log_u"])
    g = epsilon * np.asarray(log["log_v"])
    shift = -float(np.dot(a, f))
    f = f + shift
    g = g - shift
    return plan, f, g

## Localized one-dimensional marginals

We use twelve bins, about one and a half times the size of the earlier visual, and two localized Gaussian-mixture histograms.  The panels show `K`, then two row-normalization and two column-normalization steps.

In [ ]:
fig_name = "sinkhorn-marginal-errors"
out = figure_dir(fig_name)

n = 12
grid, alpha_density, beta_density, a, b, C = sinkhorn_histograms(n=n)
epsilon = 0.16
K = np.exp(-C / epsilon)
K = K / K.sum()

states = [("initial.pdf", K.copy())]
P = K.copy()
for step in range(1, 3):
    P = (a / np.maximum(P.sum(axis=1), 1e-300))[:, None] * P
    states.append((f"row-{step}.pdf", P.copy()))
    P = P * (b / np.maximum(P.sum(axis=0), 1e-300))[None, :]
    states.append((f"column-{step}.pdf", P.copy()))

## Matrix with marginal strips

Matrix entries and marginal values are drawn with circles whose areas encode mass.  Hollow markers show the requested marginals; filled markers show the current marginals after the displayed scaling step.

In [ ]:
def draw_scaling_panel(P, filename):
    row = P.sum(axis=1)
    col = P.sum(axis=0)
    max_mass = max(P.max(), a.max(), b.max(), row.max(), col.max())

    fig, ax = plt.subplots(figsize=(2.12, 2.08))
    for k in range(n):
        ax.plot([-0.5, n - 0.5], [k, k], color=LIGHT_GRAY, lw=0.34, alpha=0.55, zorder=0)
        ax.plot([k, k], [-0.5, n - 0.5], color=LIGHT_GRAY, lw=0.34, alpha=0.55, zorder=0)

    yy, xx = np.mgrid[0:n, 0:n]
    sizes = 4.5 + 128 * (P.ravel() / max_mass) ** 0.72
    ax.scatter(xx.ravel(), yy.ravel(), s=sizes, marker="o", color=VIOLET, alpha=0.72, edgecolor="none", zorder=2)

    strip = n + 0.75
    ax.scatter(np.arange(n), np.full(n, -1.25), s=4.5 + 128 * (b / max_mass) ** 0.72, marker="o", facecolor="none", edgecolor=BLUE, linewidth=0.7, zorder=4)
    ax.scatter(np.arange(n), np.full(n, -0.70), s=4.5 + 128 * (col / max_mass) ** 0.72, marker="o", color=BLUE, edgecolor="none", alpha=0.85, zorder=4)
    ax.scatter(np.full(n, strip), np.arange(n), s=4.5 + 128 * (a / max_mass) ** 0.72, marker="o", facecolor="none", edgecolor=RED, linewidth=0.7, zorder=4)
    ax.scatter(np.full(n, strip - 0.55), np.arange(n), s=4.5 + 128 * (row / max_mass) ** 0.72, marker="o", color=RED, edgecolor="none", alpha=0.85, zorder=4)

    ax.set_xlim(-0.85, n + 1.05)
    ax.set_ylim(n - 0.45, -1.70)
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)


for filename, P_state in states:
    draw_scaling_panel(P_state, filename)